In [2]:
import pandas as pd
import re

# Everest Death Data from Wikipedia

Source: https://en.wikipedia.org/wiki/List_of_people_who_died_climbing_Mount_Everest


In [3]:
#import data
ever_death_wiki = pd.read_csv("data/everest_death_wikipedia.csv")
ever_death_wiki.head()

,Unnamed: 0,Date,Age,Expedition,Nationality,Cause of death,Location,Remains status,Refs
0,Alexander Mitchell Kellas,5-Jun-21,52,1921 British Mount Everest reconnaissance expe...,United Kingdom,Heart attack,En route to base camp near the Tibetan village...,"Recovered, buried near Kampa Dzong",[9][26]
1,Unknown porter,5-Jun-21,NaN,1921 British Mount Everest reconnaissance expe...,NaN,NaN,En route to base camp near the Tibetan village...,NaN,[10]
2,Dorje,7-Jun-22,NaN,1922 British Mount Everest expedition,Nepal,Avalanche,Below North Col,NaN,[9]
3,Lhakpa,7-Jun-22,NaN,1922 British Mount Everest expedition,Nepal,Avalanche,Below North Col,NaN,[9]
4,Norbu,7-Jun-22,NaN,1922 British Mount Everest expedition,Nepal,Avalanche,Below North Col,NaN,[9]


In [5]:
ever_death_wiki.info()
print(len(ever_death_wiki))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Unnamed: 0      347 non-null    object
 1   Date            346 non-null    object
 2   Age             214 non-null    object
 3   Expedition      311 non-null    object
 4   Nationality     346 non-null    object
 5   Cause of death  336 non-null    object
 6   Location        329 non-null    object
 7   Remains status  99 non-null     object
 8   Refs            346 non-null    object
dtypes: object(9)
memory usage: 24.7+ KB
350


In [224]:
#how many NANs in "Location" column?
ever_death_wiki["Location"].isna().sum()

#save to no_location df
no_location = ever_death_wiki[ever_death_wiki["Location"].isna()]
no_location.head()

,Unnamed: 0,Date,Age,Expedition,Nationality,Cause of death,Location,Remains status,Refs
15,Wang Ji,11-Apr-60,NaN,Chinese Expedition Northern Slope,China,Mountain sickness,NaN,NaN,[32][33]
21,Kunga Norbu,5-Apr-70,NaN,Japanese Skiing Expedition,Nepal,Avalanche,NaN,NaN,[40]
107,Ang Pinjo,12-Dec-89,NaN,South Korean expedition,Nepal,"Altitude sickness, heart attack",NaN,NaN,[38]
108,Rafael G—mez-Menor,14-Sep-90,NaN,NaN,Spain,Avalanche,NaN,NaN,[84]
109,Ang Sona,14-Sep-90,NaN,NaN,Nepal,Avalanche,NaN,NaN,[84]


In [225]:
#remove locations = NAN
ever_death_wiki = ever_death_wiki.dropna(subset=["Location"])
ever_death_wiki.head()
#if Nationality contains NAN, replace with "Unknown"
ever_death_wiki["Nationality"] = ever_death_wiki["Nationality"].fillna("Unknown")
ever_death_wiki.head()


#create Role Column - if Nationality contains "Nepal", Role = "Nepalese", else "Climber" 
ever_death_wiki["Role"] = ever_death_wiki["Nationality"].apply(lambda x: "Nepalese" if "Nepal" in x else "Climber")


#change name of "Unnamed: 0" column to "Name"
ever_death_wiki = ever_death_wiki.rename(columns={"Unnamed: 0": "Name"})
ever_death_wiki.head() 

,Name,Date,Age,Expedition,Nationality,Cause of death,Location,Remains status,Refs,Role
0,Alexander Mitchell Kellas,5-Jun-21,52,1921 British Mount Everest reconnaissance expe...,United Kingdom,Heart attack,En route to base camp near the Tibetan village...,"Recovered, buried near Kampa Dzong",[9][26],Climber
1,Unknown porter,5-Jun-21,NaN,1921 British Mount Everest reconnaissance expe...,Unknown,NaN,En route to base camp near the Tibetan village...,NaN,[10],Climber
2,Dorje,7-Jun-22,NaN,1922 British Mount Everest expedition,Nepal,Avalanche,Below North Col,NaN,[9],Nepalese
3,Lhakpa,7-Jun-22,NaN,1922 British Mount Everest expedition,Nepal,Avalanche,Below North Col,NaN,[9],Nepalese
4,Norbu,7-Jun-22,NaN,1922 British Mount Everest expedition,Nepal,Avalanche,Below North Col,NaN,[9],Nepalese


In [226]:
#sort location by most common
pd.set_option('display.max_rows', None)

ever_death_wiki["Location"].value_counts()

Location
Icefall                                                                 36
Base Camp                                                               27
Below North Col                                                         13
8600m N.E. Ridge                                                        10
Camp II                                                                  7
Near Summit                                                              7
N.E. Ridge                                                               7
Balcony                                                                  7
South Col                                                                7
6400m                                                                    7
8500m N.E. Ridge                                                         5
Camp IV                                                                  5
8700m N.E. Ridge                                                         5
South Summit    

In [227]:
ever_death_cleaned_locations = ever_death_wiki.copy()

#remove commas from "Location" column
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].str.replace(",", "")

#all deaths that mention "Balcony" in the location will be categorized as "Balcony"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "Balcony" if re.search("Balcony", x) else x)
#all deaths mentioning Hillary Step will be categorized as "Hillary Step"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "Hillary Step" if re.search("Hillary Step", x) else x)
#Hornbein Couloir will be categorized as "Hornbein Couloir"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "Hornbein Couloir" if re.search("Hornbein Couloir", x) else x)
#all icefall becomes Khumbu Icefall
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "Khumbu Icefall" if re.search("Icefall", x) else x)


#Routes should be organized from bottom to top -> need to parse out North Route, South Route, and West Ridge route
    #because some locations are names and some are elevations, will add evelations to front of location name

#if "Base Camp North Side" in location, add "6490m" to front of location name
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "6490m " + x if re.search("Base Camp North Side", x) else x)
# if "North Col" in location, add "7000m" to front of location name
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "7000m " + x if re.search("North Col", x) else x)
# if "Karl" in "Name", change location to 6400m Below North Col
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: "6400m Below North Col" if re.search("Karl", row["Name"]) else row["Location"], axis=1)
#if only 6400m in location, change to 6400m W Ridge
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "6400m W Ridge" if re.search("6400m", x) and not re.search("North Col", x) else x)
#if "Above" or "Descending" and "South Col" in location, change to "8000m Above South Col"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: "8000m Above South Col" if (re.search("Above", row["Location"]) or re.search("Descending", row["Location"])) and re.search("South Col", row["Location"]) else row["Location"], axis=1)
#if "Above South Col" and no numeric characters in location, change to "8000m Above South Col"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: "8000m Above South Col" if re.search("Above South Col", row["Location"]) and not re.search("\d", row["Location"]) else row["Location"], axis=1)
#locations where "South Col" appears with no numeric characters, add 7900m to front of location name
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: "7900m " + row["Location"] if re.search("South Col", row["Location"]) and not re.search("\d", row["Location"]) else row["Location"], axis=1)
#If Marty Hoey is in "Name", change location to "8000m N.E. Ridge"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: "8000m N.E. Ridge" if re.search("Marty Hoey", row["Name"]) else row["Location"], axis=1)
#if only "8000m" in location, change to "8000m W Ridge"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "8000m W Ridge" if re.search("8000m", x) and not re.search("South Col", x) and not re.search("N.E. Ridge", x) else x)
#if "Lhotse Face" or "Lhoste face" in location, change to "7400m Lhotse Face"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "7400m Lhotse Face" if re.search("Lhotse Face", x) or re.search("Lhotse face", x) else x)
#if only "Near Camp IV" in location, change to "Camp IV"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "Camp IV" if re.search("Near Camp IV", x) else x)
#if "Camp IV (south side) in tent" or "Camp IV in the tent" in location, change to "Camp IV"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "Camp IV" if re.search("Camp IV \(south side\) in tent", x) or re.search("Camp IV in the tent", x) else x)
#if only "Base camp" case senitive, change to "Base Camp"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "Base Camp" if re.search("Base camp", x) else x)
#if contains "Base Camp North Side" or "Rongbuk", chang to "Base Camp North Side"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "Base Camp North Side" if re.search("Base Camp North Side", x) or re.search("Rongbuk", x) else x)
#if "NE" in location, change that text to "N.E." same with SE to S.E.
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: re.sub(r"\bNE\b", "N.E.", x))
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: re.sub(r"\bSE\b", "S.E.", x))
#if "S.W. Face" in location and no numeric, change to "7700m S.W. Face"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: "7700m S.W. Face" if re.search("S.W. Face", row["Location"]) and not re.search("\d", row["Location"]) else row["Location"], axis=1)
#if "W ridge" or "W Ridge" in location, repalce with "West Ridge"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: re.sub(r"W Ridge|W ridge", "West Ridge", x))
#"Near summit" and "Below the Summit" becomes "Near Summit"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "Near Summit" if re.search("Near summit", x) or re.search("Below the Summit", x) else x)
#if "6900m" appears, change to "6900m W. Cwm"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "6900m W. Cwm" if re.search("6900m", x) else x)
#"North-East Ridge (approx. 8200m)" to "8200m N.E. Ridge"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations["Location"].apply(lambda x: "8200m N.E. Ridge" if re.search("North-East Ridge", x) and re.search("8200m", x) else x)


ever_death_cleaned_locations["Location"].value_counts()


Location
Khumbu Icefall                                                          46
Base Camp                                                               30
7000m Below North Col                                                   13
Balcony                                                                 13
Near Summit                                                             10
Camp IV                                                                 10
8600m N.E. Ridge                                                        10
7400m Lhotse Face                                                        8
7900m South Col                                                          8
Hillary Step                                                             8
8000m West Ridge                                                         7
N.E. Ridge                                                               7
Camp II                                                                  7
6400m West Ridge

In [228]:
#Adding Route information to deaths with only elevation information
#adding identifying info to location, which is later sorted into Route bu the categorize_route function

#in Name column:

#if Kyak Tsering in name, change location to "Khumbu Icefall"
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: "Khumbu Icefall" if re.search("Kyak Tsering", row["Name"]) else row["Location"], axis=1)
#if Kiyoshi Narita in name, add "S.E. Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " S.E. Ridge" if re.search("Kiyoshi Narita", row["Name"]) else row["Location"], axis=1)
# if Wu Zhuong Yue in name, add "N.E. Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " N.E. Ridge" if re.search("Wu Zhuong Yue", row["Name"]) else row["Location"], axis=1)
#Mick Burke in name, add "S.E. Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " S.E. Ridge" if re.search("Mick Burke", row["Name"]) else row["Location"], axis=1)
#Lhakpa Tshering in name, add "West Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " West Ridge" if re.search("Lhakpa Tshering", row["Name"]) else row["Location"], axis=1)
#Yasuo Kato or Toshiaki Kobayashi in name, add "S.E. Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " S.E. Ridge" if re.search("Yasuo Kato", row["Name"]) or re.search("Toshiaki Kobayashi", row["Name"]) else row["Location"], axis=1)
# Hironobu Kamuro or Hiroshi Yoshino in name, add "S.E. Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " S.E. Ridge" if re.search("Hironobu Kamuro", row["Name"]) or re.search("Hiroshi Yoshino", row["Name"]) else row["Location"], axis=1)
#Tony Swierzy, add "N.E. Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " N.E. Ridge" if re.search("Tony Swierzy", row["Name"]) else row["Location"], axis=1)
#Jozef Psotka, add "S.E. Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " S.E. Ridge" if re.search("Jozef Psotka", row["Name"]) else row["Location"], axis=1)
#Juanjo Navarro, add "N.E. Ridge" to location - Claude is wrong, it's not South
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " N.E. Ridge" if re.search("Juanjo Navarro", row["Name"]) else row["Location"], axis=1)
#Kiran Inder Kumar in name, add "S.E. Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " S.E. Ridge" if re.search("Kiran Inder Kumar", row["Name"]) else row["Location"], axis=1)
#Simon Burkhardt in name, add "S.E. Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " S.E. Ridge" if re.search("Simon Burkhardt", row["Name"]) else row["Location"], axis=1)
#Tsuttin Dorje, add "S.W. Face" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " S.W. Face" if re.search("Tsuttin Dorje", row["Name"]) else row["Location"], axis=1)
#if Michel Parmentier in name, add "West Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " West Ridge" if re.search("Michel Parmentier", row["Name"]) else row["Location"], axis=1)
#if Narayan Shrestha in name, add "West Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " West Ridge" if re.search("Narayan Shrestha", row["Name"]) else row["Location"], axis=1)
#Lhakpa Sonam and Pasang Temba, add "S.E. Ridge" to location - provenance of this info uncertain
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " S.E. Ridge" if re.search("Lhakpa Sonam", row["Name"]) or re.search("Pasang Temba", row["Name"]) else row["Location"], axis=1)
#Libor Kozák, add "N.E. Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " N.E. Ridge" if re.search("Libor", row["Name"]) else row["Location"], axis=1)
#Peter Kinloch, add "N.E. Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " N.E. Ridge" if re.search("Peter Kinloch", row["Name"]) else row["Location"], axis=1)
#Rick Hitch, add "S.E. Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " S.E. Ridge" if re.search("Rick Hitch", row["Name"]) else row["Location"], axis=1)
#Ang Phurba Sherpa, add "S.E. Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " S.E. Ridge" if re.search("Ang Phurba Sherpa", row["Name"]) else row["Location"], axis=1)
#Robin Haynes Fisher, add "S.E. Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " S.E. Ridge" if re.search("Robin Haynes Fisher", row["Name"]) else row["Location"], axis=1)
#Shrinivas Sainis Dattatraya or Joshua Cheruiyot Kirui or Nawang Sherpa, add "S.E. Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " S.E. Ridge" if re.search("Shrinivas Sainis Dattatraya", row["Name"]) or re.search("Joshua Cheruiyot Kirui", row["Name"]) or re.search("Nawang Sherpa", row["Name"]) else row["Location"], axis=1)
#Zangbu, add "S.E. Ridge" to location
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " S.E. Ridge" if re.search("Zangbu", row["Name"]) else row["Location"], axis=1)


#special fixes:
#if 17-May-07 in "Date", add "N.E. Ridge" to location - this is Shinichi Ishii number 2
ever_death_cleaned_locations["Location"] = ever_death_cleaned_locations.apply(lambda row: row["Location"] + " N.E. Ridge" if re.search("17-May-07", row["Date"]) else row["Location"], axis=1)



In [229]:
#adding a new "Route" column to categorize locations into North Route, South Route, and West Ridge
def categorize_route(location):
    #if "North" or "N.E." in location, return "North"
    if re.search("North", location) or re.search("N\.E\.", location) or re.search("Hillary Step", location) or re.search("Hornbein Couloir", location) or re.search("Norton Couloir", location) or re.search("north", location) or re.search("Tibet", location):
        return "North"
    #if "W." or "S.W." or "Lhola camp" or "West Ridge" in location, return "West Ridge" 
    elif  re.search("Lhola camp", location) or re.search("West Ridge", location):
        return "West Ridge"
    #if "South", "S.E.", "Kumbu Icefall", "Base Camp" with nothing else, return "South"
    #added balcony and Lhotse
    #added S.W. Face, which is closer to S.E. Ridge route than West Ridge route
    elif re.search("South", location) or re.search("S\.W\.", location) or re.search("Cwm", location) or re.search("south", location) or re.search("Lhotse", location) or re.search("Balcony", location) or re.search("S\.E\.", location) or re.search("Khumbu Icefall", location) or (re.search("Base Camp", location) and not re.search("North", location)):
        return "South"
    #else return "Other"
    else:
        return "Other"
    
ever_death_cleaned_locations["Route"] = ever_death_cleaned_locations["Location"].apply(categorize_route)

#if in "Other" and "Camp" in location, change route to "South"
ever_death_cleaned_locations["Route"] = ever_death_cleaned_locations.apply(lambda row: "South" if row["Route"] == "Other" and re.search("Camp", row["Location"]) else row["Route"], axis=1)

In [230]:
ever_death_cleaned_locations.groupby(["Route","Role"])["Location"].value_counts()

Route       Role      Location                                                            
North       Climber   8600m N.E. Ridge                                                        11
                      Hillary Step                                                             7
                      8500m N.E. Ridge                                                         6
                      7000m Below North Col                                                    5
                      8700m N.E. Ridge                                                         5
                      N.E. Ridge                                                               5
                      7800m N.E. Ridge                                                         3
                      8200m N.E. Ridge                                                         3
                      8750m N.E. Ridge                                                         3
                      Base Camp Nort

In [245]:
#what names have "7700m S.W. Face N.E. Ridge" in location?
ever_death_cleaned_locations[ever_death_cleaned_locations["Location"] == "7700m S.W. Face N.E. Ridge"]["Name"]

208    Lee Hyun-jo
212    Oh Hee-joon
Name: Name, dtype: object

In [231]:
ever_death_cleaned_locations.groupby(["Route","Role"])["Location"].value_counts().to_csv("data/value_counts.csv")

In [232]:
everest_death_unknown = ever_death_cleaned_locations[ever_death_cleaned_locations["Route"] == "Other"]
everest_death_unknown.head()
#export other to csv
# everest_death_unknown.to_csv("data/everest_unknown_route.csv", index=False)

,Name,Date,Age,Expedition,Nationality,Cause of death,Location,Remains status,Refs,Role,Route


In [233]:
#locations where "South Col" appears with no numeric characters
ever_death_cleaned_locations[ever_death_cleaned_locations["Location"].str.contains("South Col") & ~ever_death_cleaned_locations["Location"].str.contains("\d")]["Location"].value_counts()

Series([], Name: count, dtype: int64)

In [234]:
#look at all "Location" that start with a non numeric character
ever_death_cleaned_locations[~ever_death_cleaned_locations["Location"].str.match("^\d")]["Location"].value_counts()

Location
Khumbu Icefall                                                          47
Base Camp                                                               29
Balcony                                                                 13
Near Summit S.E. Ridge                                                  10
Camp IV                                                                 10
Hillary Step                                                             8
N.E. Ridge                                                               7
Camp II                                                                  7
Base Camp North Side                                                     5
South Summit                                                             5
Hornbein Couloir                                                         3
Camp I                                                                   3
Above Camp III                                                           2
En route to base

In [235]:
ever_death_cleaned_locations["Location"].value_counts()

#locations grouped by nationality
ever_death_cleaned_locations.groupby("Role")["Location"].value_counts()



Role      Location                                                            
Climber   Balcony                                                                 13
          Base Camp                                                               13
          8600m N.E. Ridge                                                        11
          Khumbu Icefall                                                           8
          7900m South Col                                                          7
          8000m West Ridge                                                         7
          Camp IV                                                                  7
          Hillary Step                                                             7
          Near Summit S.E. Ridge                                                   7
          8500m N.E. Ridge                                                         6
          7000m Below North Col                                        

## Everest Expedition Data for Routes

In [236]:
routes = pd.read_csv("data/EverestExpeditions_HimalayanDatabase.csv")
routes.head()


,Yr/Seas,Host,Nationalities,Leader (s),Route(s),Result,Smtrs,Dead,Exped ID
0,1921 Spr,China,"UK, Canada",C. K. Howard-Bury,"W, N & E sides",Did not Climb,NaN,2.0,EVER-211-01
1,1922 Spr,China,"UK, Australia",Charles G. Bruce,N Col-N Face,"Illness, AMS",NaN,7.0,EVER-221-01
2,1924 Spr,China,UK,Edward F. Norton,N Col-N Face (Norton to 8570m) / N Col-NE Ridg...,Accident,NaN,4.0,EVER-241-01
3,1933 Spr,China,UK,Hugh Ruttledge,N Col-N Face,Route Difficulty,NaN,NaN,EVER-331-01
4,1934 Spr,China,UK,Maurice Wilson,N Col,Accident,NaN,1.0,EVER-341-01


In [237]:
routes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2396 entries, 0 to 2395
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Yr/Seas         2396 non-null   object 
 1   Host            2396 non-null   object 
 2   Nationalities   2396 non-null   object 
 3   Leader (s)      2389 non-null   object 
 4   Route(s)        2395 non-null   object 
 5   Result          2396 non-null   object 
 6   Smtrs           1536 non-null   float64
 7   Dead            240 non-null    float64
 8   Exped ID        2396 non-null   object 
dtypes: float64(2), object(7)
memory usage: 168.6+ KB


In [238]:
routes.iloc[3,7]

np.float64(nan)

In [239]:
#column names?
routes.columns
#remove spaces from column names
routes.columns = routes.columns.str.replace(" ", "")
routes.columns
#for "Route(s)", replace / with AND-OR
routes["Route(s)"] = routes["Route(s)"].str.replace("/", " AND-OR ")


In [240]:
#drop rows where "Dead" is nan
routes = routes.dropna(subset=["Dead"])
routes.head(10)

,Yr/Seas,Host,Nationalities,Leader(s),Route(s),Result,Smtrs,Dead,ExpedID
0,1921 Spr,China,"UK, Canada",C. K. Howard-Bury,"W, N & E sides",Did not Climb,NaN,2.0,EVER-211-01
1,1922 Spr,China,"UK, Australia",Charles G. Bruce,N Col-N Face,"Illness, AMS",NaN,7.0,EVER-221-01
2,1924 Spr,China,UK,Edward F. Norton,N Col-N Face (Norton to 8570m) AND-OR N Col-...,Accident,NaN,4.0,EVER-241-01
4,1934 Spr,China,UK,Maurice Wilson,N Col,Accident,NaN,1.0,EVER-341-01
13,1952 Aut,Nepal,Switzerland,Gabriel Chevalley,S Col-SE Ridge,Bad Weather,NaN,1.0,EVER-523-01
19,1960 Spr,China,China,Shi Zhang-Chun,N Col-NE Ridge,Success,3.0,1.0,EVER-601-02
20,1962 Spr,Nepal,India,John Dias,S Col-SE Ridge,Bad Weather,NaN,1.0,EVER-621-01
22,1963 Spr,Nepal,"USA, UK",Norman Dyhrenfurth,S Col-SE Ridge AND-OR W Cwm-W Ridge-N Face (...,Success,6.0,1.0,EVER-631-01
23,1964 Spr,China,China,Xu Jing,N Col-NE Ridge (training),Other,NaN,1.0,EVER-641-01
26,1966 Spr,China,China,Xu Jing,"N Col (training, recon)",Accident,NaN,1.0,EVER-661-01


In [241]:
#count number of instances of each unique route in "Route(s)"
routes["Route(s)"].value_counts()


Route(s)
S Col-SE Ridge                                                                                                                                    127
N Col-NE Ridge                                                                                                                                     59
SW Face                                                                                                                                             4
NE Ridge                                                                                                                                            3
Lho La-W Ridge                                                                                                                                      2
N Face (Great Couloir)                                                                                                                              2
N Face (Japanese & Hornbein Couloirs)                                                      

In [242]:
#number of unique routes and number of deaths for each route
route_deaths = routes.groupby("Route(s)")["Dead"].sum()
route_deaths.sort_values(ascending=False)

Route(s)
S Col-SE Ridge                                                                                                                                    178.0
N Col-NE Ridge                                                                                                                                     70.0
Lho La-W Ridge                                                                                                                                      8.0
N Col-N Face                                                                                                                                        7.0
S Col (skiing)                                                                                                                                      6.0
Khumbutse-W Ridge-N Face (Hornbein Couloir)                                                                                                         5.0
SW Face                                                                        

In [243]:
#total deaths
total_deaths = routes["Dead"].sum()
print(total_deaths)

344.0
